In [2]:
!pip install pmdarima 

In [1]:
# =============================================================================
# Model 5 — SARIMA (Optimised — full London, auto order selection)
# Task    : Predict Crime_Count per LSOA based on historical time series
#
# Strategy:
#   - auto_arima selects best (p,d,q)(P,D,Q,12) per LSOA via AIC
#   - Parallel fitting across all CPU cores (joblib)
#   - Full London LSOA set — no sampling
#   - Refit on full 2020-2023 series, forecast all 12 months of 2024
# =============================================================================

import pandas as pd
import numpy as np
import warnings
import time
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from joblib import Parallel, delayed
from tqdm import tqdm

warnings.filterwarnings("ignore")

# Try importing pmdarima for auto_arima — install if missing
try:
    from pmdarima import auto_arima
    AUTO_ARIMA_AVAILABLE = True
    print("pmdarima available — using auto_arima for best order selection")
except ImportError:
    AUTO_ARIMA_AVAILABLE = False
    print("pmdarima not found. Installing...")
    import subprocess
    subprocess.run(["pip", "install", "pmdarima", "--break-system-packages", "-q"])
    try:
        from pmdarima import auto_arima
        AUTO_ARIMA_AVAILABLE = True
        print("pmdarima installed successfully")
    except ImportError:
        AUTO_ARIMA_AVAILABLE = False
        print("pmdarima install failed — falling back to fixed order (1,1,1)(1,1,1,12)")

# =============================================================================
# 1. LOAD DATA
# =============================================================================
df = pd.read_csv("./crime_per_capita_df_corrected.csv")
df = df.sort_values(["LSOA_Code", "Year", "Month"]).reset_index(drop=True)

print(f"\nDataset shape : {df.shape}")
print(f"Date range    : {df['Year'].min()} to {df['Year'].max()}")
print(f"Unique LSOAs  : {df['LSOA_Code'].nunique()}")

# =============================================================================
# 2. CONFIGURATION
# =============================================================================
TRAIN_YEARS  = [2020, 2021, 2022, 2023]
TEST_YEAR    = 2024
N_FORECAST   = 12          # forecast all 12 months of 2024
MIN_TRAIN_OBS= 24          # minimum months needed to fit SARIMA reliably
N_JOBS       = -1          # -1 = use ALL CPU cores

# auto_arima search space — best order found per LSOA via AIC minimisation
AUTO_ARIMA_CONFIG = {
    "start_p"        : 0,
    "max_p"          : 3,       # AR terms 0–3
    "start_q"        : 0,
    "max_q"          : 3,       # MA terms 0–3
    "d"              : None,    # auto-detect differencing order via KPSS test
    "max_d"          : 2,
    "start_P"        : 0,
    "max_P"          : 2,       # seasonal AR 0–2
    "start_Q"        : 0,
    "max_Q"          : 2,       # seasonal MA 0–2
    "D"              : None,    # auto-detect seasonal differencing
    "max_D"          : 1,
    "m"              : 12,      # monthly seasonality
    "seasonal"       : True,
    "information_criterion": "aic",   # minimise AIC for order selection
    "stepwise"       : True,    # stepwise search — much faster than exhaustive
    "suppress_warnings": True,
    "error_action"   : "ignore",
    "trace"          : False,
}

# Fallback fixed order if auto_arima fails for a specific LSOA
FALLBACK_ORDER          = (1, 1, 1)
FALLBACK_SEASONAL_ORDER = (1, 1, 1, 12)

train_df = df[df["Year"].isin(TRAIN_YEARS)].copy()
test_df  = df[df["Year"] == TEST_YEAR].copy()
lsoa_list= df["LSOA_Code"].unique()

print(f"\nTotal LSOAs to fit : {len(lsoa_list)}")
print(f"Cores available    : {__import__('os').cpu_count()}")
print(f"Parallel jobs      : {'all cores' if N_JOBS == -1 else N_JOBS}")
print(f"Order selection    : {'auto_arima (AIC)' if AUTO_ARIMA_AVAILABLE else 'fixed (1,1,1)(1,1,1,12)'}")

# =============================================================================
# 3. STATIONARITY AUDIT — sample 10 LSOAs
# Guides how aggressively we need to difference
# =============================================================================
print("\n--- Stationarity audit (10 LSOAs) ---")
n_stationary = 0
for lsoa in lsoa_list[:10]:
    series = train_df[train_df["LSOA_Code"] == lsoa]["Crime_Count"].values
    if len(series) < 4:
        continue
    p_val = adfuller(series, autolag="AIC")[1]
    status = "stationary" if p_val < 0.05 else "non-stationary"
    if p_val < 0.05:
        n_stationary += 1
    print(f"  {lsoa}  p={p_val:.4f}  {status}")
print(f"\n{n_stationary}/10 series are already stationary")
print("(non-stationary series will be corrected by differencing inside SARIMA)")

# =============================================================================
# 4. SINGLE LSOA FITTING FUNCTION
# Called in parallel for each LSOA
# Returns dict of results or failure info
# =============================================================================
def fit_sarima_lsoa(lsoa):
    try:
        # Training series
        lsoa_train = (train_df[train_df["LSOA_Code"] == lsoa]
                      .sort_values(["Year","Month"])["Crime_Count"]
                      .values.astype(float))

        # Test rows
        lsoa_test = (test_df[test_df["LSOA_Code"] == lsoa]
                     .sort_values("Month"))

        # Skip if insufficient data
        if len(lsoa_train) < MIN_TRAIN_OBS:
            return {"lsoa": lsoa, "status": "skip_data",
                    "rows": 0, "actuals": [], "forecasts": [], "order": None}
        if len(lsoa_test) == 0:
            return {"lsoa": lsoa, "status": "skip_notest",
                    "rows": 0, "actuals": [], "forecasts": [], "order": None}

        # --- Order selection ---
        if AUTO_ARIMA_AVAILABLE:
            try:
                auto_model = auto_arima(lsoa_train, **AUTO_ARIMA_CONFIG)
                best_order          = auto_model.order
                best_seasonal_order = auto_model.seasonal_order
            except Exception:
                best_order          = FALLBACK_ORDER
                best_seasonal_order = FALLBACK_SEASONAL_ORDER
        else:
            best_order          = FALLBACK_ORDER
            best_seasonal_order = FALLBACK_SEASONAL_ORDER

        # --- Fit SARIMA with selected order ---
        model = SARIMAX(
            lsoa_train,
            order=best_order,
            seasonal_order=best_seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False,
            concentrate_scale=True,   # faster fitting
        )
        fitted = model.fit(
            disp=False,
            method="lbfgs",           # L-BFGS-B optimiser — fastest for SARIMA
            maxiter=200,
            optim_score=None,
        )

        # --- Forecast 12 months ahead ---
        forecast = np.clip(fitted.forecast(steps=N_FORECAST), 0, None)

        # --- Collect per-month results ---
        actuals   = []
        forecasts = []
        months    = []
        test_months = lsoa_test["Month"].values

        for month in test_months:
            if month <= N_FORECAST:
                actual_vals = lsoa_test[lsoa_test["Month"] == month]["Crime_Count"].values
                if len(actual_vals) > 0:
                    actuals.append(float(actual_vals[0]))
                    forecasts.append(float(forecast[month - 1]))
                    months.append(int(month))

        return {
            "lsoa"     : lsoa,
            "status"   : "ok",
            "rows"     : len(actuals),
            "actuals"  : actuals,
            "forecasts": forecasts,
            "months"   : months,
            "order"    : str(best_order),
            "seasonal" : str(best_seasonal_order),
            "aic"      : round(fitted.aic, 2),
        }

    except Exception as e:
        return {"lsoa": lsoa, "status": f"error: {str(e)[:80]}",
                "rows": 0, "actuals": [], "forecasts": [], "order": None}

# =============================================================================
# 5. PARALLEL FITTING — all LSOAs simultaneously across all cores
# =============================================================================
print(f"\nFitting SARIMA for all {len(lsoa_list)} LSOAs in parallel...")
print("This will take approximately 15–40 minutes depending on CPU count.\n")

start_time = time.time()

raw_results = Parallel(n_jobs=N_JOBS, verbose=5, backend="loky")(
    delayed(fit_sarima_lsoa)(lsoa) for lsoa in lsoa_list
)

elapsed = time.time() - start_time
print(f"\nParallel fitting complete in {elapsed/60:.1f} minutes")

# =============================================================================
# 6. AGGREGATE RESULTS
# =============================================================================
all_actuals   = []
all_forecasts = []
all_lsoas     = []
all_months    = []
orders_used   = []
failed        = []
skipped       = []

for r in raw_results:
    if r["status"] == "ok" and r["rows"] > 0:
        all_actuals.extend(r["actuals"])
        all_forecasts.extend(r["forecasts"])
        all_lsoas.extend([r["lsoa"]] * r["rows"])
        all_months.extend(r.get("months", []))
        orders_used.append(r["order"])
    elif r["status"].startswith("skip"):
        skipped.append(r["lsoa"])
    else:
        failed.append((r["lsoa"], r["status"]))

print(f"\nSuccessfully predicted : {len(lsoa_list) - len(failed) - len(skipped)} LSOAs")
print(f"Skipped (data issues)  : {len(skipped)}")
print(f"Failed (errors)        : {len(failed)}")
print(f"Total prediction rows  : {len(all_actuals):,}")

if failed:
    print(f"\nFirst 5 failures:")
    for lsoa, reason in failed[:5]:
        print(f"  {lsoa}: {reason}")

# Most common orders selected by auto_arima
if orders_used:
    from collections import Counter
    order_counts = Counter(orders_used).most_common(5)
    print(f"\nTop 5 most common orders selected by auto_arima:")
    for order, count in order_counts:
        print(f"  {order} — {count} LSOAs")

# =============================================================================
# 7. EVALUATION
# =============================================================================
if len(all_actuals) == 0:
    print("\nERROR: No predictions were generated. Check data and configuration.")
else:
    y_true = np.array(all_actuals)
    y_pred = np.array(all_forecasts)

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

    print(f"\n{'='*55}")
    print(f"  SARIMA — auto order per LSOA (AIC selection)")
    print(f"{'='*55}")
    print(f"  LSOAs fitted    : {len(lsoa_list) - len(failed) - len(skipped)}")
    print(f"  Prediction rows : {len(y_true):,}")
    print(f"  MAE             : {mae:.4f}")
    print(f"  RMSE            : {rmse:.4f}")
    print(f"  R²              : {r2:.4f}")
    print(f"  MAPE            : {mape:.2f}%")
    print(f"  Runtime         : {elapsed/60:.1f} minutes")

    sarima_result = {
        "Model": "SARIMA — auto order AIC (per LSOA)",
        "MAE"  : round(mae, 4),
        "RMSE" : round(rmse, 4),
        "R2"   : round(r2, 4),
        "MAPE" : round(mape, 2),
    }

    # =========================================================================
    # 8. ERROR ANALYSIS
    # =========================================================================
    result_df = pd.DataFrame({
        "LSOA_Code" : all_lsoas,
        "Month"     : all_months,
        "Actual"    : y_true,
        "Predicted" : np.round(y_pred, 2),
        "Abs_Error" : np.abs(y_true - y_pred),
        "Pct_Error" : np.abs(y_true - y_pred) / (y_true + 1e-9) * 100,
    })

    # Error by crime count bucket
    result_df["Crime_Bucket"] = pd.cut(
        result_df["Actual"], bins=[0, 5, 15, 30, 50, 9999],
        labels=["0–5", "6–15", "16–30", "31–50", "50+"]
    )
    bucket_err = (result_df.groupby("Crime_Bucket", observed=True)["Abs_Error"]
                  .agg(["mean", "count"])
                  .rename(columns={"mean": "Avg_MAE", "count": "N_rows"}))
    print("\nError by crime count bucket:")
    print(bucket_err.to_string())

    # Error by month — does SARIMA struggle with specific seasons?
    monthly_err = (result_df.groupby("Month")["Abs_Error"]
                   .mean().reset_index()
                   .rename(columns={"Abs_Error": "Avg_MAE"}))
    print("\nAverage error by month (1=Jan … 12=Dec):")
    print(monthly_err.to_string(index=False))

    # Worst LSOAs
    worst = (result_df.groupby("LSOA_Code")["Abs_Error"]
             .mean().sort_values(ascending=False).head(10))
    print("\nTop 10 hardest LSOAs:")
    print(worst.to_string())

    # Sample predictions
    print("\n--- Sample Predictions vs Actual (first 12 rows) ---")
    print(result_df.head(12)[
        ["LSOA_Code","Month","Actual","Predicted","Abs_Error","Pct_Error"]
    ].to_string(index=False))

    # Save detailed predictions
    result_df.to_csv("sarima_predictions.csv", index=False)
    print("\nDetailed predictions saved to sarima_predictions.csv")

    # =========================================================================
    # 9. SAVE TO MODEL COMPARISON FILE
    # =========================================================================
    new_row = pd.DataFrame([sarima_result])

    try:
        existing = pd.read_csv("model_comparison.csv")
        combined = pd.concat([existing, new_row], ignore_index=True)
    except FileNotFoundError:
        combined = new_row

    combined.to_csv("model_comparison.csv", index=False)

    print("\n\n--- Updated model_comparison.csv ---")
    print(combined[["Model","MAE","RMSE","R2","MAPE"]].to_string(index=False))

pmdarima available — using auto_arima
Dataset shape: (304336, 39)
Unique LSOAs: 12415

Running SARIMA on all LSOAs...

Valid predictions: 55793

SARIMA Results
MAE : 58563.2812
RMSE: 13829364.9095
R2  : -139356543280.7553
MAPE: 177494.75 %
Runtime: 16.01 minutes

Saved predictions → sarima_predictions.csv


In [1]:
# =============================================================================
# Model 5 — SARIMA (Optimised — full London, auto order selection)
# =============================================================================

import pandas as pd
import numpy as np
import warnings
import time
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from joblib import Parallel, delayed
from tqdm import tqdm

warnings.filterwarnings("ignore")

# Try importing pmdarima for auto_arima — install if missing
try:
    from pmdarima import auto_arima
    AUTO_ARIMA_AVAILABLE = True
    print("pmdarima available — using auto_arima for best order selection")
except ImportError:
    AUTO_ARIMA_AVAILABLE = False
    print("pmdarima not found. Installing...")
    import subprocess
    subprocess.run(["pip", "install", "pmdarima", "--break-system-packages", "-q"])
    try:
        from pmdarima import auto_arima
        AUTO_ARIMA_AVAILABLE = True
        print("pmdarima installed successfully")
    except ImportError:
        AUTO_ARIMA_AVAILABLE = False
        print("pmdarima install failed — falling back to fixed order (1,1,1)(1,1,1,12)")

# =============================================================================
# 1. LOAD DATA
# =============================================================================
df = pd.read_csv("./crime_per_capita_df_corrected.csv")
df = df.sort_values(["LSOA_Code", "Year", "Month"]).reset_index(drop=True)

print(f"\nDataset shape : {df.shape}")
print(f"Date range    : {df['Year'].min()} to {df['Year'].max()}")
print(f"Unique LSOAs  : {df['LSOA_Code'].nunique()}")

# =============================================================================
# 2. CONFIGURATION
# =============================================================================
TRAIN_YEARS  = [2020, 2021, 2022, 2023]
TEST_YEAR    = 2024
N_FORECAST   = 12
MIN_TRAIN_OBS= 24
N_JOBS       = -1

AUTO_ARIMA_CONFIG = {
    "start_p"        : 0,
    "max_p"          : 3,
    "start_q"        : 0,
    "max_q"          : 3,
    "d"              : None,
    "max_d"          : 2,
    "start_P"        : 0,
    "max_P"          : 2,
    "start_Q"        : 0,
    "max_Q"          : 2,
    "D"              : None,
    "max_D"          : 1,
    "m"              : 12,
    "seasonal"       : True,
    "information_criterion": "aic",
    "stepwise"       : True,
    "suppress_warnings": True,
    "error_action"   : "ignore",
    "trace"          : False,
}

FALLBACK_ORDER          = (1, 1, 1)
FALLBACK_SEASONAL_ORDER = (1, 1, 1, 12)

train_df = df[df["Year"].isin(TRAIN_YEARS)].copy()
test_df  = df[df["Year"] == TEST_YEAR].copy()
lsoa_list= df["LSOA_Code"].unique()

print(f"\nTotal LSOAs to fit : {len(lsoa_list)}")
print(f"Cores available    : {__import__('os').cpu_count()}")
print(f"Parallel jobs      : {'all cores' if N_JOBS == -1 else N_JOBS}")
print(f"Order selection    : {'auto_arima (AIC)' if AUTO_ARIMA_AVAILABLE else 'fixed (1,1,1)(1,1,1,12)'}")

# =============================================================================
# 3. STATIONARITY AUDIT
# =============================================================================
print("\n--- Stationarity audit (10 LSOAs) ---")
n_stationary = 0
for lsoa in lsoa_list[:10]:
    series = train_df[train_df["LSOA_Code"] == lsoa]["Crime_Count"].values
    if len(series) < 4:
        continue
    p_val = adfuller(series, autolag="AIC")[1]
    status = "stationary" if p_val < 0.05 else "non-stationary"
    if p_val < 0.05:
        n_stationary += 1
    print(f"  {lsoa}  p={p_val:.4f}  {status}")
print(f"\n{n_stationary}/10 series are already stationary")

# =============================================================================
# 4. SINGLE LSOA FITTING FUNCTION
# =============================================================================
def fit_sarima_lsoa(lsoa):
    try:

        lsoa_train = (train_df[train_df["LSOA_Code"] == lsoa]
                      .sort_values(["Year","Month"])["Crime_Count"]
                      .values.astype(float))

        lsoa_test = (test_df[test_df["LSOA_Code"] == lsoa]
                     .sort_values("Month"))

        if len(lsoa_train) < MIN_TRAIN_OBS:
            return {"lsoa": lsoa, "status": "skip_data",
                    "rows": 0, "actuals": [], "forecasts": [], "order": None}

        if len(lsoa_test) == 0:
            return {"lsoa": lsoa, "status": "skip_notest",
                    "rows": 0, "actuals": [], "forecasts": [], "order": None}

        if AUTO_ARIMA_AVAILABLE:
            try:
                auto_model = auto_arima(lsoa_train, **AUTO_ARIMA_CONFIG)
                best_order          = auto_model.order
                best_seasonal_order = auto_model.seasonal_order
            except Exception:
                best_order          = FALLBACK_ORDER
                best_seasonal_order = FALLBACK_SEASONAL_ORDER
        else:
            best_order          = FALLBACK_ORDER
            best_seasonal_order = FALLBACK_SEASONAL_ORDER

        model = SARIMAX(
            lsoa_train,
            order=best_order,
            seasonal_order=best_seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False,
            concentrate_scale=True,
        )

        fitted = model.fit(
            disp=False,
            method="lbfgs",
            maxiter=200,
            optim_score=None,
        )

        # ==============================
        # MODIFIED FORECAST LINE (ONLY CHANGE)
        # ==============================
        forecast = fitted.forecast(steps=N_FORECAST)

        forecast = np.nan_to_num(forecast, nan=0.0, posinf=0.0, neginf=0.0)

        forecast = np.clip(forecast, 0, np.percentile(lsoa_train, 99) * 3)
        # ==============================

        actuals   = []
        forecasts = []
        months    = []
        test_months = lsoa_test["Month"].values

        for month in test_months:
            if month <= N_FORECAST:
                actual_vals = lsoa_test[lsoa_test["Month"] == month]["Crime_Count"].values
                if len(actual_vals) > 0:
                    actuals.append(float(actual_vals[0]))
                    forecasts.append(float(forecast[month - 1]))
                    months.append(int(month))

        return {
            "lsoa"     : lsoa,
            "status"   : "ok",
            "rows"     : len(actuals),
            "actuals"  : actuals,
            "forecasts": forecasts,
            "months"   : months,
            "order"    : str(best_order),
            "seasonal" : str(best_seasonal_order),
            "aic"      : round(fitted.aic, 2),
        }

    except Exception as e:
        return {"lsoa": lsoa, "status": f"error: {str(e)[:80]}",
                "rows": 0, "actuals": [], "forecasts": [], "order": None}

# =============================================================================
# 5. PARALLEL FITTING
# =============================================================================
print(f"\nFitting SARIMA for all {len(lsoa_list)} LSOAs in parallel...")
start_time = time.time()

raw_results = Parallel(n_jobs=N_JOBS, verbose=5, backend="loky")(
    delayed(fit_sarima_lsoa)(lsoa) for lsoa in lsoa_list
)

elapsed = time.time() - start_time
print(f"\nParallel fitting complete in {elapsed/60:.1f} minutes")

# =============================================================================
# 6. AGGREGATE RESULTS
# =============================================================================
all_actuals   = []
all_forecasts = []
all_lsoas     = []
all_months    = []
orders_used   = []
failed        = []
skipped       = []

for r in raw_results:
    if r["status"] == "ok" and r["rows"] > 0:
        all_actuals.extend(r["actuals"])
        all_forecasts.extend(r["forecasts"])
        all_lsoas.extend([r["lsoa"]] * r["rows"])
        all_months.extend(r.get("months", []))
        orders_used.append(r["order"])
    elif r["status"].startswith("skip"):
        skipped.append(r["lsoa"])
    else:
        failed.append((r["lsoa"], r["status"]))

print(f"\nSuccessfully predicted : {len(lsoa_list) - len(failed) - len(skipped)} LSOAs")
print(f"Skipped (data issues)  : {len(skipped)}")
print(f"Failed (errors)        : {len(failed)}")
print(f"Total prediction rows  : {len(all_actuals):,}")

if failed:
    print(f"\nFirst 5 failures:")
    for lsoa, reason in failed[:5]:
        print(f"  {lsoa}: {reason}")

# Most common orders selected by auto_arima
if orders_used:
    from collections import Counter
    order_counts = Counter(orders_used).most_common(5)
    print(f"\nTop 5 most common orders selected by auto_arima:")
    for order, count in order_counts:
        print(f"  {order} — {count} LSOAs")

# =============================================================================
# 7. EVALUATION
# =============================================================================
if len(all_actuals) == 0:
    print("\nERROR: No predictions were generated. Check data and configuration.")
else:
    y_true = np.array(all_actuals)
    y_pred = np.array(all_forecasts)

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

    print(f"\n{'='*55}")
    print(f"  SARIMA — auto order per LSOA (AIC selection)")
    print(f"{'='*55}")
    print(f"  LSOAs fitted    : {len(lsoa_list) - len(failed) - len(skipped)}")
    print(f"  Prediction rows : {len(y_true):,}")
    print(f"  MAE             : {mae:.4f}")
    print(f"  RMSE            : {rmse:.4f}")
    print(f"  R²              : {r2:.4f}")
    print(f"  MAPE            : {mape:.2f}%")
    print(f"  Runtime         : {elapsed/60:.1f} minutes")

    sarima_result = {
        "Model": "SARIMA — auto order AIC (per LSOA)",
        "MAE"  : round(mae, 4),
        "RMSE" : round(rmse, 4),
        "R2"   : round(r2, 4),
        "MAPE" : round(mape, 2),
    }

    # =========================================================================
    # 8. ERROR ANALYSIS
    # =========================================================================
    result_df = pd.DataFrame({
        "LSOA_Code" : all_lsoas,
        "Month"     : all_months,
        "Actual"    : y_true,
        "Predicted" : np.round(y_pred, 2),
        "Abs_Error" : np.abs(y_true - y_pred),
        "Pct_Error" : np.abs(y_true - y_pred) / (y_true + 1e-9) * 100,
    })

    # Error by crime count bucket
    result_df["Crime_Bucket"] = pd.cut(
        result_df["Actual"], bins=[0, 5, 15, 30, 50, 9999],
        labels=["0–5", "6–15", "16–30", "31–50", "50+"]
    )
    bucket_err = (result_df.groupby("Crime_Bucket", observed=True)["Abs_Error"]
                  .agg(["mean", "count"])
                  .rename(columns={"mean": "Avg_MAE", "count": "N_rows"}))
    print("\nError by crime count bucket:")
    print(bucket_err.to_string())

    # Error by month — does SARIMA struggle with specific seasons?
    monthly_err = (result_df.groupby("Month")["Abs_Error"]
                   .mean().reset_index()
                   .rename(columns={"Abs_Error": "Avg_MAE"}))
    print("\nAverage error by month (1=Jan … 12=Dec):")
    print(monthly_err.to_string(index=False))

    # Worst LSOAs
    worst = (result_df.groupby("LSOA_Code")["Abs_Error"]
             .mean().sort_values(ascending=False).head(10))
    print("\nTop 10 hardest LSOAs:")
    print(worst.to_string())

    # Sample predictions
    print("\n--- Sample Predictions vs Actual (first 12 rows) ---")
    print(result_df.head(12)[
        ["LSOA_Code","Month","Actual","Predicted","Abs_Error","Pct_Error"]
    ].to_string(index=False))

    # Save detailed predictions
    result_df.to_csv("sarima_predictions.csv", index=False)
    print("\nDetailed predictions saved to sarima_predictions.csv")

    # =========================================================================
    # 9. SAVE TO MODEL COMPARISON FILE
    # =========================================================================
    new_row = pd.DataFrame([sarima_result])

    try:
        existing = pd.read_csv("model_comparison.csv")
        combined = pd.concat([existing, new_row], ignore_index=True)
    except FileNotFoundError:
        combined = new_row

    combined.to_csv("model_comparison.csv", index=False)

    print("\n\n--- Updated model_comparison.csv ---")
    print(combined[["Model","MAE","RMSE","R2","MAPE"]].to_string(index=False))

pmdarima available — using auto_arima for best order selection

Dataset shape : (304336, 39)
Date range    : 2020 to 2024
Unique LSOAs  : 12415

Total LSOAs to fit : 12415
Cores available    : 16
Parallel jobs      : all cores
Order selection    : auto_arima (AIC)

--- Stationarity audit (10 LSOAs) ---
  E01000001  p=0.0000  stationary
  E01000002  p=0.0002  stationary
  E01000003  p=0.9789  non-stationary
  E01000005  p=0.8235  non-stationary
  E01000006  p=0.0000  stationary
  E01000007  p=0.0000  stationary
  E01000008  p=0.0001  stationary
  E01000009  p=0.7702  non-stationary
  E01000011  p=0.7502  non-stationary
  E01000012  p=0.0002  stationary

6/10 series are already stationary

Fitting SARIMA for all 12415 LSOAs in parallel...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 15 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:    7.0s
[Parallel(n_jobs=-1)]: Done 132 tasks      | elapsed:   18.7s
[Parallel(n_jobs=-1)]: Done 258 tasks      | elapsed:   33.7s
[Parallel(n_jobs=-1)]: Done 420 tasks      | elapsed:   53.7s
[Parallel(n_jobs=-1)]: Done 618 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 852 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-1)]: Done 1122 tasks      | elapsed:  2.2min
[Parallel(n_jobs=-1)]: Done 1428 tasks      | elapsed:  2.9min
[Parallel(n_jobs=-1)]: Done 1770 tasks      | elapsed:  3.6min
[Parallel(n_jobs=-1)]: Done 2148 tasks      | elapsed:  4.4min
[Parallel(n_jobs=-1)]: Done 2562 tasks      | elapsed:  5.3min
[Parallel(n_jobs=-1)]: Done 3012 tasks      | elapsed:  6.3min
[Parallel(n_jobs=-1)]: Done 3498 tasks      | elapsed:  7.3min
[Parallel(n_jobs=-1)]: Done 4020 tasks      | elapsed:  8.2min
[Parallel(n_jobs=-1)]: Done 4578 tasks      | e


Parallel fitting complete in 16.2 minutes

Successfully predicted : 3376 LSOAs
Skipped (data issues)  : 7738
Failed (errors)        : 1301
Total prediction rows  : 40,341

First 5 failures:
  E01000001: error: not enough values to unpack (expected 2, got 0)
  E01000006: error: not enough values to unpack (expected 2, got 0)
  E01000013: error: not enough values to unpack (expected 2, got 0)
  E01000015: error: not enough values to unpack (expected 2, got 0)
  E01000020: error: not enough values to unpack (expected 2, got 0)

Top 5 most common orders selected by auto_arima:
  (0, 1, 1) — 1116 LSOAs
  (1, 0, 0) — 731 LSOAs
  (0, 0, 1) — 337 LSOAs
  (0, 0, 0) — 306 LSOAs
  (2, 0, 0) — 143 LSOAs

  SARIMA — auto order per LSOA (AIC selection)
  LSOAs fitted    : 3376
  Prediction rows : 40,341
  MAE             : 8.0395
  RMSE            : 16.2988
  R²              : 0.7972
  MAPE            : 54.52%
  Runtime         : 16.2 minutes

Error by crime count bucket:
                Avg_MAE  N